# Test `discover_skill_headers`
This notebook tests the `discover_skill_headers` function from `src/aux_functions.py`.
The function scans `skills/*/SKILL.md` files and returns a dict of the form:
```
{ skill_name: [skill_description, skill_path] }
```

In [9]:
import sys
from pathlib import Path

# Make sure the project root is on the path
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: C:\Users\Leal\AI_Projects\analytics_content_agent


In [2]:
from src.aux_functions import discover_skill_headers

skills = discover_skill_headers()
print(f"Found {len(skills)} skill(s)\n")
skills

Found 4 skill(s)



{'analyzing-time-series': ['Comprehensive diagnostic analysis of time series data. Use when users provide CSV time series data and want to understand its characteristics before forecasting - stationarity, seasonality, trend, forecastability, and transform recommendations.',
  'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/analyzing-time-series/SKILL.md'],
 'crm-omnichannel-analysis': ['Unified diagnostic analysis of CRM marketing data. Integrates online (Google Analytics) and offline (Physical Stores) data to calculate retention KPIs, LTV, Churn, and cross-channel purchasing behavior. Designed to produce structured outputs for the .docx reporting skill.',
  'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/crm-omnichannel-analysis/SKILL.md'],
 'docx': ["Use this skill whenever the user wants to create, read, edit, or manipulate Word documents (.docx files). Triggers include: any mention of 'Word doc', 'word document', '.docx', or requests to produce professional docu

In [ ]:

C:\Users\Leal\AI_Projects\analytics_content_agent\skills
C:/Users/Leal/AI_Projects/analytics_content_agent/skills

WindowsPath('C:/Users/Leal/AI_Projects/analytics_content_agent/skills')

In [15]:
import json
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import re


def _parse_yaml_header(markdown: str) -> Optional[Dict[str, str]]:
    """
    Parse only the top YAML front matter block.

    This deliberately supports the simple `key: value` shape used by the local
    Skill files and avoids importing a YAML dependency just for metadata.
    """

    lines = markdown.splitlines()
    if not lines or lines[0].strip() != "---":
        return None

    header_lines: List[str] = []
    for line in lines[1:]:
        if line.strip() == "---":
            break
        header_lines.append(line)
    else:
        return None

    header: Dict[str, str] = {}
    for line in header_lines:
        if ":" not in line:
            continue

        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")

        if key:
            header[key] = value

    return header or None

def _skills_catalog() -> Dict[str, List[str]]:
    """
    Load Skill metadata from project/Skills/*/SKILL.md using YAML headers only.

    Returns a dict mapping skill_name -> [skill_description, skill_path].
    """

    skill_files: List[Path] = []
    seen: set[Path] = set()
    skills_root = Path(".").resolve() / "skills"

    for skill_file in skills_root.glob("*/SKILL.md"):
        resolved = skill_file.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        skill_files.append(skill_file)

    skills: Dict[str, List[str]] = {}
    for skill_file in sorted(skill_files):
        try:
            markdown = skill_file.read_text(encoding="utf-8")
        except OSError:
            continue

        header = _parse_yaml_header(markdown)
        if not header:
            continue

        name = header.get("name", "").strip()
        description = header.get("description", "").strip()
        if name and description:
            skills[name] = [description, skill_file.as_posix()]

    return skills

def _skills_environment(skills: dict[str, List[str]]) -> List[dict[str, str]]:
    environment_skills: List[dict[str, str]] = []
    for name, skill_info in skills.items():
        description, skill_path = skill_info
        path = Path(skill_path)
        environment_skills.append(
            {
                "name": name,
                "description": description,
                "path": path.parent.as_posix() if path.name == "SKILL.md" else path.as_posix(),
            }
        )

    return environment_skills


In [18]:
skills = _skills_catalog()
skills_ajusted = _skills_environment(skills)

In [20]:

skills_ajusted

[{'name': 'analyzing-time-series',
  'description': 'Comprehensive diagnostic analysis of time series data. Use when users provide CSV time series data and want to understand its characteristics before forecasting - stationarity, seasonality, trend, forecastability, and transform recommendations.',
  'path': 'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/analyzing-time-series'},
 {'name': 'crm-omnichannel-analysis',
  'description': 'Unified diagnostic analysis of CRM marketing data. Integrates online (Google Analytics) and offline (Physical Stores) data to calculate retention KPIs, LTV, Churn, and cross-channel purchasing behavior. Designed to produce structured outputs for the .docx reporting skill.',
  'path': 'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/crm-omnichannel-analysis'},
 {'name': 'docx',
  'description': "Use this skill whenever the user wants to create, read, edit, or manipulate Word documents (.docx files). Triggers include: any mention of 'Word

In [49]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-mini",
    input="""use some skill to create an generical docs in my filesistem folder called files""",
    tools=[
        {
            "type": "shell",
            "environment": {
                "type": "local",
                "skills": [
                    {
                        'name': 'docx',
                        'description': "Use this skill whenever the user wants to create, read, edit, or manipulate Word documents (.docx files). Triggers include: any mention of 'Word doc', 'word document', '.docx', or requests to produce professional documents with formatting like tables of contents, headings, page numbers, or letterheads. Also use when extracting or reorganizing content from .docx files, inserting or replacing images in documents, performing find-and-replace in Word files, working with tracked changes or comments, or converting content into a polished Word document. If the user asks for a 'report', 'memo', 'letter', 'template', or similar deliverable as a Word or .docx file, use this skill. Do NOT use for PDFs, spreadsheets, Google Docs, or general coding tasks unrelated to document generation.",
                        'path': 'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/docx'
                    }
                ],
            },
        }
    ],
   
)

print(response)

Response(id='resp_0a8c926c8949de940069fcfc7fc808819d8ed2637d32b22f19', created_at=1778187391.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0a8c926c8949de940069fcfc809584819d8a9b00aa13486a31', content=[ResponseOutputText(annotations=[], text='Using the **docx** skill because you asked to create a document file in your filesystem.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='commentary'), ResponseFunctionShellToolCall(id='sh_0a8c926c8949de940069fcfc80c480819dade2b30e2a8a6c44', action=Action(commands=["cat 'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/docx/SKILL.md'"], max_output_length=12000, timeout_ms=10000), call_id='call_pWo71ClnscG3lF3Rr1sPLuXv', environment=None, status='completed', type='shell_call', created_by=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionShellTo

In [47]:
print(response)

Response(id='resp_030bd1f571aaf65d0069fcfa1abba48195a310cb395dbaf7d5', created_at=1778186778.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_030bd1f571aaf65d0069fcfa1b587c8195a99fec7ca36de15c', content=[ResponseOutputText(annotations=[], text='Using the **docx** skill because you asked to create a document file in your filesystem.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='commentary'), ResponseFunctionShellToolCall(id='sh_030bd1f571aaf65d0069fcfa1b8fbc8195abb4405d3cf2c746', action=Action(commands=["cat 'C:/Users/Leal/AI_Projects/analytics_content_agent/skills/docx/SKILL.md'"], max_output_length=12000, timeout_ms=10000), call_id='call_zIwNwSWqoe1n30CQ0C7qtED4', environment=None, status='completed', type='shell_call', created_by=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionShellTo

In [45]:
for output in response.output:
    print(output)

ResponseOutputMessage(id='msg_0ecab2947b5be6670069fcf799675c819cb7a9c669056cb41e', content=[ResponseOutputText(annotations=[], text='Using the **docx** skill because you asked to create a document file in your filesystem.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='commentary')
ResponseFunctionShellToolCall(id='sh_0ecab2947b5be6670069fcf79996b8819cbe32127d4f68ebe9', action=Action(commands=['type C:/Users/Leal/AI_Projects/analytics_content_agent/skills/docx/SKILL.md && echo. && dir C:\\Users\\Leal\\AI_Projects\\analytics_content_agent\\skills\\docx'], max_output_length=20000, timeout_ms=10000), call_id='call_QikXxQzwTUq50TSKFqvCpIMP', environment=None, status='completed', type='shell_call', created_by=None)


In [41]:
print(response.output_text)

Using the docx skill because you asked to create a Word document.


In [1]:
!pip install -r requirements.txt

   ---------------------------------------- 0.0/683.6 kB ? eta -:--:--
   ---------------------------------------- 683.6/683.6 kB 13.2 MB/s  0:00:00
Using cached python_multipart-0.0.27-py3-none-any.whl (29 kB)
   ---------------------------------------- 0.0/15.2 MB ? eta -:--:--
   ------------------------ --------------- 9.2/15.2 MB 43.9 MB/s eta 0:00:01
   ---------------------------------------- 15.2/15.2 MB 47.9 MB/s  0:00:00

   -- -------------------------------------  1/17 [sqlglot]
   -- -------------------------------------  1/17 [sqlglot]
   -- -------------------------------------  1/17 [sqlglot]
   -- -------------------------------------  1/17 [sqlglot]
   -- -------------------------------------  1/17 [sqlglot]
   ------- --------------------------------  3/17 [pyparsing]
   ----------- ----------------------------  5/17 [protobuf]
   ----------- ----------------------------  5/17 [protobuf]
   -------------- -------------------------  6/17 [oauthlib]
   ----------------